### 1: Завдання

Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних

In [1]:
import urllib.request
import pandas as pd
import os
from datetime import datetime
from IPython.display import display

def download_data(province_id, folder_path="vhi_data"):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

    for file in os.listdir(folder_path):
        if file.startswith(f"vhi_{province_id}_"):
            print(f"Регіон {province_id} вже існує.")
            return
            
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    now = datetime.now().strftime("%Y%m%d%H%M%S")
    file_name = f"vhi_{province_id}_{now}.csv"
    full_path = os.path.join(folder_path, file_name)

    try:
        urllib.request.urlretrieve(url, full_path)
        print(f"Завантажено: {file_name}")
    except Exception as e:
        print(f"{province_id}: {e}")

for i in range(1, 28):
    download_data(i)

Завантажено: vhi_1_20260313183843.csv
Завантажено: vhi_2_20260313183848.csv
Завантажено: vhi_3_20260313183849.csv
Завантажено: vhi_4_20260313183852.csv
Завантажено: vhi_5_20260313183853.csv
Завантажено: vhi_6_20260313183854.csv
Завантажено: vhi_7_20260313183855.csv
Завантажено: vhi_8_20260313183856.csv
Завантажено: vhi_9_20260313183857.csv
Завантажено: vhi_10_20260313183858.csv
Завантажено: vhi_11_20260313183859.csv
Завантажено: vhi_12_20260313183900.csv
Завантажено: vhi_13_20260313183901.csv
Завантажено: vhi_14_20260313183902.csv
Завантажено: vhi_15_20260313183903.csv
Завантажено: vhi_16_20260313183904.csv
Завантажено: vhi_17_20260313183905.csv
Завантажено: vhi_18_20260313183906.csv
Завантажено: vhi_19_20260313183907.csv
Завантажено: vhi_20_20260313183908.csv
Завантажено: vhi_21_20260313183909.csv
Завантажено: vhi_22_20260313183910.csv
Завантажено: vhi_23_20260313183911.csv
Завантажено: vhi_24_20260313183912.csv
Завантажено: vhi_25_20260313183914.csv
Завантажено: vhi_26_20260313183915

### 2: Завдання

Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області

In [2]:
def data_cleaning(folder_path="vhi_data"):
    all_dfs = []
    names = {1: 'Cherkasy', 2: 'Chernihiv', 3: 'Chernivtsi', 4: 'Crimea', 5: 'Dnipropetrovsk', 6: 'Donetsk', 7: 'Ivano-Frankivsk', 8: 'Kharkiv', 9: 'Kherson', 10: 'Khmelnytskyy', 11: 'Kyiv', 12: 'Kyiv City', 13: 'Kirovohrad', 14: 'Luhansk', 15: 'Lviv', 16: 'Mykolayiv', 17: 'Odessa', 18: 'Poltava', 19: 'Rivne', 20: 'Sevastopol', 21: 'Sumy', 22: 'Ternopil', 23: 'Transcarpathia', 24: 'Vinnytsya', 25: 'Volyn', 26: 'Zaporizhzhya', 27: 'Zhytomyr'}
    
    for file_name in os.listdir(folder_path):
            
        province_id = int(file_name.split('_')[1])
        full_path = os.path.join(folder_path, file_name)
    
        df = pd.read_csv(full_path, header=1, names=['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty'], skipinitialspace=True)
        df = df.drop(columns=['empty'], errors='ignore')
        df['year'] = df['year'].astype(str).str.replace('<tt><pre>', '', regex=False)
        df = df[df['year'] != '']
        df = df.replace(-1, pd.NA)
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        df = df.dropna()
        df['year'] = df['year'].astype(int)
        df['week'] = df['week'].astype(int)
        df['province_id'] = province_id
        df['province_name'] = names.get(province_id, "Unknown")
        all_dfs.append(df)
        
    main_df = pd.concat(all_dfs, ignore_index=True)
    return main_df

df = data_cleaning()
display(df.head())

,year,week,SMN,SMT,VCI,TCI,VHI,province_id,province_name
0,1982,1,0.059,258.24,51.11,48.78,49.95,10,Khmelnytskyy
1,1982,2,0.063,261.53,55.89,38.20,47.04,10,Khmelnytskyy
2,1982,3,0.063,263.45,57.30,32.69,44.99,10,Khmelnytskyy
3,1982,4,0.061,265.10,53.96,28.62,41.29,10,Khmelnytskyy
4,1982,5,0.058,266.42,46.87,28.57,37.72,10,Khmelnytskyy


### 3: Завдання

Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька).

In [3]:
def reindex_provinces(df):
    mapping_dict = {
        1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21, 
        11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16, 
        20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
    }
    
    names_dict = {
        1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька', 5: 'Житомирська',
        6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська', 9: 'Київська', 10: 'Кіровоградська',
        11: 'Луганська', 12: 'Львівська', 13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська',
        16: 'Рівненська', 17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська',
        21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська', 25: 'Республіка Крим',
        26: 'м. Київ', 27: 'м. Севастополь'
    }
    
    df_copy = df.copy()
    df_copy['province_id'] = df_copy['province_id'].map(mapping_dict)
    df_copy['province_name'] = df_copy['province_id'].map(names_dict)
    
    return df_copy

df_reindexed = reindex_provinces(df)
display(df_reindexed.head())

,year,week,SMN,SMT,VCI,TCI,VHI,province_id,province_name
0,1982,1,0.059,258.24,51.11,48.78,49.95,21,Хмельницька
1,1982,2,0.063,261.53,55.89,38.20,47.04,21,Хмельницька
2,1982,3,0.063,263.45,57.30,32.69,44.99,21,Хмельницька
3,1982,4,0.061,265.10,53.96,28.62,41.29,21,Хмельницька
4,1982,5,0.058,266.42,46.87,28.57,37.72,21,Хмельницька


### 4: Завдання

4.1. Реалізувати процедури для формування вибірок (Ряд VHI для області за вказаний рік)

In [56]:
def get_vhi_for_year(df, province_name, year):
    filtered_df = df[(df['province_name'] == province_name) & (df['year'] == year)][['year', 'VHI', 'province_name']]
    return filtered_df
    
display(get_vhi_for_year(df_reindexed, 'Чернігівська', 2006).head())    

,year,VHI,province_name
42732,2006,51.23,Чернігівська
42733,2006,51.83,Чернігівська
42734,2006,50.74,Чернігівська
42735,2006,51.52,Чернігівська
42736,2006,51.13,Чернігівська


4.2. Реалізувати процедури для формування вибірок (Ряд VHI за вказаний діапазон років для вказаних областей)

In [4]:
def get_vhi_for_range_of_years(df, province_names, from_year, to_year):
    filtered_df = df[(df['province_name'].isin(province_names)) & (df['year'] >= from_year) & (df['year'] <= to_year)][['year', 'VHI', 'province_name']]
    return filtered_df
    
display(get_vhi_for_range_of_years(df_reindexed, ['Чернігівська', 'Львівська', 'Київська'], 2006, 2010).head()) 

,year,VHI,province_name
3384,2006,46.09,Київська
3385,2006,48.27,Київська
3386,2006,48.84,Київська
3387,2006,50.32,Київська
3388,2006,50.61,Київська


4.3. Реалізувати процедури для формування вибірок (Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани)

In [5]:
def get_vhi_stats(df, province_names, years):
    filtered_df = df[(df['province_name'].isin(province_names)) & df['year'].isin(years)]
    stats = {
        'provinces': [province_names],
        'years': [years],
        'VHI_min': [filtered_df['VHI'].min()],
        'VHI_max': [filtered_df['VHI'].max()],
        'VHI_mean': [filtered_df['VHI'].mean()],
        'VHI_median': [filtered_df['VHI'].median()]
    }
    return pd.DataFrame(stats)
    
display(get_vhi_stats(df_reindexed, ['Чернігівська', 'Львівська', 'Київська'], [2006, 2008, 2010]))

,provinces,years,VHI_min,VHI_max,VHI_mean,VHI_median
0,"[Чернігівська, Львівська, Київська]","[2006, 2008, 2010]",21.83,74.45,47.582094,47.01
